# 🤖 Demo · Agente PrediMant

**Mantenimiento predictivo industrial** · Variante 2 · UTH 2026.4

Autor: **Kenny Xavier Lanza Rios** ([@kennylanza1509](https://github.com/kennylanza1509))

Este cuaderno recorre, paso a paso, cómo el agente pasa de **datos crudos de sensores** a un **diagnóstico con acción recomendada y notificación por correo**.

> Ejecuta las celdas en orden (Shift+Enter).

## Arquitectura

```
datos (sensores) -> histórico CSV -> predicción (tendencia)
                                          |
                                          v
          RAG (cita norma) <- severidad -> diagnóstico -> 📧 correo
```

- **Cerebro (razonamiento):** Claude Code (sin API key externa).
- **Herramientas (código Python):** generar datos, predecir, consultar RAG, notificar.

In [ ]:
# Preparación: hacemos visibles las carpetas src/ y tools/ para poder importarlas.
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
sys.path.insert(0, os.path.abspath('../tools'))
print('Rutas listas ✓')

## Paso 1 · Clases POO — `SensorVirtual`

Un sensor que genera lecturas sintéticas y se va **degradando** con el tiempo
(valor base + tendencia + ruido).

In [ ]:
from modelos import SensorVirtual, Equipo, EventoFalla

sensor = SensorVirtual('TEMP', '°C', valor_base=70.0, ruido=0.8, degradacion=0.4, umbral_falla=90.0)
for _ in range(5):
    print(sensor.leer())

## Paso 2 · Clase `Equipo` — agrupa varios sensores

Un motor real tiene varios sensores. `Equipo` los lee juntos y calcula el
**estado general**: NORMAL / ALERTA / FALLA.

In [ ]:
motor = Equipo('MOTOR-01', 'motor')
motor.agregar_sensor(SensorVirtual('TEMP', '°C',   70.0, ruido=0.8,  degradacion=0.40, umbral_falla=90.0))
motor.agregar_sensor(SensorVirtual('VIB',  'mm/s',  2.0, ruido=0.15, degradacion=0.08, umbral_falla=7.0))
motor.agregar_sensor(SensorVirtual('CORR', 'A',    12.0, ruido=0.30, degradacion=0.05, umbral_falla=18.0))

for paso in range(1, 6):
    reporte = motor.monitorear()
    valores = {l['tag']: l['valor'] for l in reporte['lecturas']}
    print(paso, valores, '->', reporte['estado'])

## Paso 3 · Generar el histórico (CSV)

Simulamos 60 pasos y guardamos todo en `data/lecturas.csv` y `data/eventos.csv`.

In [ ]:
import generar_datos
generar_datos.main()

## Paso 4 · Predicción por tendencia (la tool)

`predecir_falla` ajusta una **recta** a cada sensor y estima en cuántos pasos
cruzará su umbral.

In [ ]:
from predecir_falla import cargar_series, predecir_sensor, UMBRALES, RUTA_CSV

series = cargar_series(RUTA_CSV)
for s, puntos in series.items():
    print(predecir_sensor(s, puntos, UMBRALES[s]))

## Paso 5 · RAG — el agente cita la norma técnica

Búsqueda TF-IDF sobre `data/conocimiento/`. Recupera el fragmento más relevante.

In [ ]:
from rag import MotorRAG

rag = MotorRAG()
resultado = rag.consultar('vibracion creciente zona C motor', k=1)[0]
print('Fuente:', resultado['nombre'], '| score:', round(resultado['score'], 3))
print('-' * 50)
print(resultado['texto'][:400])

## Paso 6 · El agente completo — diagnóstico

Junta todo: predice, clasifica severidad y cita la referencia técnica.
(No enviamos correo en el cuaderno para no saturar la bandeja.)

In [ ]:
import agente

severidad, hallazgos = agente.diagnosticar()
agente.imprimir_diagnostico(severidad, hallazgos)

ref = agente.referencia_tecnica(hallazgos)
if ref:
    sensor, fuente, resumen = ref
    print(f'\nReferencia técnica (RAG · {sensor} · {fuente}):\n  {resumen}')

## Paso 7 · Notificación por correo (vista previa)

Así se ve el correo que el agente envía cuando hay riesgo. El envío real se hace
desde la terminal con `python tools/notificar.py` (usa credenciales de `.env`).

In [ ]:
import notificar

asunto, cuerpo = notificar.construir_cuerpo()
print('Asunto:', asunto)
print('-' * 50)
print(cuerpo)

## Cierre

El agente **PrediMant** cubre el flujo completo: datos → predicción → severidad →
RAG → diagnóstico → correo. Todo en Python con librería estándar y Claude Code
como cerebro. ✅